In [1]:
MODEL_NAME = "RyanDDD/empathy-mental-health-reddit-IP"

# Requirements

In [2]:
import pandas as pd
from datasets import Dataset
import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer, AutoConfig
from sklearn.model_selection import train_test_split
from datasets import load_dataset
from torch import nn
import types
import os
from tqdm import tqdm

In [3]:
# get rid of wandb api key request in google colab
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

# Data Preparation

In [4]:
# load model and tokenizer
model_name = MODEL_NAME
tokenizer = AutoTokenizer.from_pretrained(model_name)
config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(model_name, trust_remote_code=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/448 [00:00<?, ?B/s]

modeling_empathy.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/RyanDDD/empathy-mental-health-reddit-IP:
- modeling_empathy.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.01G [00:00<?, ?B/s]

Some weights of the model checkpoint at RyanDDD/empathy-mental-health-reddit-IP were not used when initializing EmpathyModel: ['model.responder_encoder.roberta.pooler.dense.bias', 'model.responder_encoder.roberta.pooler.dense.weight', 'model.seeker_encoder.roberta.pooler.dense.bias', 'model.seeker_encoder.roberta.pooler.dense.weight']
- This IS expected if you are initializing EmpathyModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing EmpathyModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [5]:
# load each input for the 3 empathy classifiers
ER_df = pd.read_csv('input-emotional-reactions-reddit.csv')
IP_df = pd.read_csv('input-interpretations-reddit.csv')
EX_df = pd.read_csv('input-explorations-reddit.csv')

In [6]:
ER_df['label'] = ER_df['level'].astype(int)
IP_df['label'] = IP_df['level'].astype(int)
EX_df['label'] = EX_df['level'].astype(int)

In [7]:
# drop unused columns
columns_to_drop = ['id', 'rationale_labels_trimmed', 'response_post_masked']

ER_df = ER_df.drop(columns=columns_to_drop)
IP_df = IP_df.drop(columns=columns_to_drop)
EX_df = EX_df.drop(columns=columns_to_drop)

In [8]:
ER_train_df, ER_test_df = train_test_split(ER_df, test_size=0.2, random_state=42)
IP_train_df, IP_test_df = train_test_split(IP_df, test_size=0.2, random_state=42)
EX_train_df, EX_test_df = train_test_split(EX_df, test_size=0.2, random_state=42)

In [9]:
ER_train_dataset = Dataset.from_pandas(ER_train_df)
ER_test_dataset = Dataset.from_pandas(ER_test_df)

IP_train_dataset = Dataset.from_pandas(IP_train_df)
IP_test_dataset = Dataset.from_pandas(IP_test_df)

EX_train_dataset = Dataset.from_pandas(EX_train_df)
EX_test_dataset = Dataset.from_pandas(EX_test_df)

In [10]:
# taken from huggingface model card
def tokenize_function(examples):
    encoded_sp = tokenizer(examples["seeker_post"], max_length=64, padding="max_length", truncation=True)
    encoded_rp = tokenizer(examples["response_post"], max_length=64, padding="max_length", truncation=True)
    return {
        "input_ids_SP": encoded_sp["input_ids"],
        "attention_mask_SP": encoded_sp["attention_mask"],
        "input_ids_RP": encoded_rp["input_ids"],
        "attention_mask_RP": encoded_rp["attention_mask"],
        "labels": examples["label"]
    }

ER_tokenized_train = ER_train_dataset.map(tokenize_function, batched=True, remove_columns=ER_train_dataset.column_names)
ER_tokenized_test = ER_test_dataset.map(tokenize_function, batched=True, remove_columns=ER_test_dataset.column_names)

IP_tokenized_train = IP_train_dataset.map(tokenize_function, batched=True, remove_columns=IP_train_dataset.column_names)
IP_tokenized_test = IP_test_dataset.map(tokenize_function, batched=True, remove_columns=IP_test_dataset.column_names)

EX_tokenized_train = EX_train_dataset.map(tokenize_function, batched=True, remove_columns=EX_train_dataset.column_names)
EX_tokenized_test = EX_test_dataset.map(tokenize_function, batched=True, remove_columns=EX_test_dataset.column_names)

Map:   0%|          | 0/2220 [00:00<?, ? examples/s]

Map:   0%|          | 0/556 [00:00<?, ? examples/s]

Map:   0%|          | 0/2220 [00:00<?, ? examples/s]

Map:   0%|          | 0/556 [00:00<?, ? examples/s]

Map:   0%|          | 0/2220 [00:00<?, ? examples/s]

Map:   0%|          | 0/556 [00:00<?, ? examples/s]

# GLOBAL VARIABLES

In [11]:
# change variables for each empathy metric
MODEL_CHECKPOINT_FOLDER = "./IP_model_checkpoints"
TRAIN_DATASET= IP_tokenized_train
TEST_DATASET = IP_tokenized_test
FINAL_MODEL = "./IP_final_model"
MODEL_PATH = "/content/IP_final_model"
REPO = "RyanDDD/empathy-mental-health-reddit-IP"
TEST_DF = IP_test_df
OUTPUT_CSV = "IP-input-interpretation-test-reddit.csv"

# Training Individual Classifiers

In [12]:
def safe_forward(self, **kwargs):
    # filter for only the four columns it needs
    filtered = {k: v for k, v in kwargs.items() if k in ALLOWED_KEYS}
    return original_forward(**filtered)

In [13]:
# fix for model columns
ALLOWED_KEYS = ['input_ids_SP', 'input_ids_RP', 'attention_mask_SP', 'attention_mask_RP']

if not hasattr(model, "_is_patched"):
    original_forward = model.forward
    model.forward = types.MethodType(safe_forward, model)
    model._is_patched = True

In [50]:
# define training args, only works with colab A100 gpu, G4 doesn't work
from transformers import TrainingArguments, Trainer, default_data_collator

training_args = TrainingArguments(
    output_dir=MODEL_CHECKPOINT_FOLDER,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    remove_unused_columns=False,
    label_names=["labels"],
    warmup_ratio= 0.01,
    bf16=True,
    tf32=True,
    per_device_train_batch_size=16,
    num_train_epochs=1,
    report_to="none"                 # don't log or else wandb api request will pop up
)

In [51]:
# create a new trainer since we are using a bi-encoder regular one won't work
class EmpathyTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs[0]
        loss_fct = nn.CrossEntropyLoss()
        return (loss_fct(logits, labels), outputs) if return_outputs else loss_fct(logits, labels)

In [52]:
trainer = EmpathyTrainer(
    model=model,
    args=training_args,
    train_dataset=TRAIN_DATASET,
    eval_dataset=TEST_DATASET,
    data_collator=default_data_collator
)

trainer.train()

Could not estimate the number of tokens of the input, floating-point operations will not be computed


Epoch,Training Loss,Validation Loss
1,No log,0.589135


TrainOutput(global_step=139, training_loss=0.3856714646593272, metrics={'train_runtime': 25.7347, 'train_samples_per_second': 86.265, 'train_steps_per_second': 5.401, 'total_flos': 0.0, 'train_loss': 0.3856714646593272, 'epoch': 1.0})

In [53]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.589135468006134, 'eval_runtime': 1.8725, 'eval_samples_per_second': 296.925, 'eval_steps_per_second': 37.383, 'epoch': 1.0}


In [54]:
trainer.save_model(FINAL_MODEL)

# Prediction on Labeled Dataset

In [55]:
def batch_predict(seeker_texts, response_texts, batch_size=32):
    predictions = []

    for i in tqdm(range(0, len(seeker_texts), batch_size)):
        batch_s = seeker_texts[i:i+batch_size].tolist()
        batch_r = response_texts[i:i+batch_size].tolist()

        # tokenize
        sp = tokenizer(batch_s, max_length=512, padding="max_length", truncation=True, return_tensors="pt")
        rp = tokenizer(batch_r, max_length=512, padding="max_length", truncation=True, return_tensors="pt")

        inputs = {
            "input_ids_SP": sp["input_ids"].to("cuda"),
            "attention_mask_SP": sp["attention_mask"].to("cuda"),
            "input_ids_RP": rp["input_ids"].to("cuda"),
            "attention_mask_RP": rp["attention_mask"].to("cuda")
        }

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs[0] # empathy level logits
            batch_preds = torch.argmax(logits, dim=-1).cpu().numpy()
            predictions.extend(batch_preds)

    return predictions

In [56]:
# google colab path
model_path = MODEL_PATH
original_repo = REPO

# get tokenizer and model
print("Loading tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(original_repo)
model = AutoModel.from_pretrained(model_path, trust_remote_code=True)
model.to("cuda")
model.eval()

# fix the column bug
ALLOWED_KEYS = ['input_ids_SP', 'input_ids_RP', 'attention_mask_SP', 'attention_mask_RP']
original_forward = model.forward

model.forward = types.MethodType(safe_forward, model)

Loading tokenizer and model...


In [57]:
df = TEST_DF

# run on every post in csv
results = batch_predict(df['seeker_post'], df['response_post'])

# save
mapping = {0: "Low", 1: "Medium", 2: "High"}
df['predicted_level'] = [mapping[p] for p in results]
df.to_csv(OUTPUT_CSV, index=False)

df[['seeker_post', 'response_post', 'level', 'predicted_level']].head()

100%|██████████| 18/18 [00:02<00:00,  8.47it/s]


,seeker_post,response_post,level,predicted_level
879,How do you manage to be happy with a full time...,Sometimes it's not a matter of choice. But if ...,0,Low
1989,Told my friend I want to CTB. Literally just t...,"Wow, if you need to talk about it, you can DM me.",0,Low
889,"Circle of my life. I'm depressed and fat, so I...",Same thing happened to me. I used to work for ...,2,High
912,Why does everyone i see seem so perfect. Why e...,I can relate to some degree. You've got it rig...,1,Low
1596,Stating the obvious. I had told my therapist e...,"You made your therapist speechless, time for a...",0,Low


In [58]:
from sklearn.metrics import accuracy_score, classification_report

label_to_idx = {'Low': 0, 'Medium': 1, 'High': 2}
y_pred = df['predicted_level'].map(label_to_idx)
y_true = df['level'].astype(int)

accuracy = accuracy_score(y_true, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

print("\nDetailed Performance:")
print(classification_report(y_true, y_pred, target_names=['Low', 'Medium', 'High']))

Model Accuracy: 85.07%

Detailed Performance:
              precision    recall  f1-score   support

         Low       0.88      0.87      0.87       291
      Medium       0.00      0.00      0.00        21
        High       0.82      0.91      0.86       244

    accuracy                           0.85       556
   macro avg       0.57      0.59      0.58       556
weighted avg       0.82      0.85      0.83       556



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Prediction on Unlabeled Dataset

In [71]:
PREDICT_DF = 'posts_eval_set_unknown_race_gpt4.csv'
PREDICT_OUTPUT = "IP_posts_eval_set_unknown_race_gpt4.csv"

In [72]:
# google colab path
model_path = MODEL_PATH
original_repo = REPO

# get tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(original_repo)
model = AutoModel.from_pretrained(model_path, trust_remote_code=True)
model.to("cuda")
model.eval()

# fix the column bug
ALLOWED_KEYS = ['input_ids_SP', 'input_ids_RP', 'attention_mask_SP', 'attention_mask_RP']
original_forward = model.forward

model.forward = types.MethodType(safe_forward, model)

In [73]:
# load dataset
df = pd.read_csv(PREDICT_DF)

# format in the right way
df = df.rename(columns={"post": "seeker_post", "gpt4_response": "response_post"})

In [74]:
# run on every post in csv
print(f"Processing {len(df)} rows...")
results = batch_predict(df['seeker_post'], df['response_post'])

# save
mapping = {0: "Low", 1: "Medium", 2: "High"}
df['predicted_level'] = [mapping[p] for p in results]
df.to_csv(PREDICT_OUTPUT, index=False)

label_to_idx = {'Low': 0, 'Medium': 1, 'High': 2}
temp = df['predicted_level'].map(label_to_idx)
df['predicted_level'] = temp
df[['seeker_post', 'response_post', 'predicted_level']].head()
df.to_csv(PREDICT_OUTPUT, index=False)



Processing 520 rows...


100%|██████████| 17/17 [00:02<00:00,  7.23it/s]
